In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [21]:
full_data = pd.read_csv("../CleanData/full_clean_data_without_imput.csv")
full_data.columns

Index(['gid', 'visteam', 'hometeam', 'site', 'daynight', 'attendance', 'temp',
       'windspeed', 'vruns', 'hruns', 'wteam', 'lteam', 'Home_Away',
       'score_diff', 'left_on_base', 'bat_doubles', 'bat_triples',
       'bat_home_runs', 'bat_stolen_bases', 'passed_balls_allowed',
       'errors_committed', 'stolen_bases', 'date', 'yearID', 'teamID',
       'salary'],
      dtype='object')

In [22]:
# these columns cannot be used in training
# gid : is a unique game id
# vruns : visiting team runs
# hruns : home team runs
# wteam : winner team
# lteam : lost team
# teamID : team id
# date: date of game, it does not matter
# visteam, hometeam : visiting and home team, does not matter
drop_cols = ["gid", "visteam", "hometeam", "date", "vruns", "hruns", "wteam", "lteam", "teamID"]
full_data = full_data.drop(drop_cols, axis=1)

In [23]:
one_hot = pd.get_dummies(full_data["site"], drop_first = True) #dropping one column to avoid collinearty issue
full_data = pd.concat([full_data,one_hot], axis=1)

In [24]:
y = full_data["score_diff"]
x = full_data.drop("score_diff", axis=1)

In [25]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify = x["Home_Away"])

In [26]:
def impute_na_data(merged_2):
    averages = merged_2.groupby("site")[["attendance", "temp", "windspeed"]].mean().reset_index()

    attendance_col_index = merged_2.columns.get_loc("attendance")
    temp_col_index = merged_2.columns.get_loc("temp")
    windsp_col_index = merged_2.columns.get_loc("windspeed")
    site_col_index = merged_2.columns.get_loc("site")

    for i in range(merged_2.shape[0]):
        attendance = merged_2.iloc[i,attendance_col_index]
        temp = merged_2.iloc[i,temp_col_index]
        windspeed = merged_2.iloc[i,windsp_col_index]
        site = merged_2.iloc[i,site_col_index]
    
        if attendance == 0:
            merged_2.iloc[i,attendance_col_index] = int(averages.loc[averages["site"] == site,"attendance"].values[0])
        if windspeed == 0:
            merged_2.iloc[i,windsp_col_index] = averages.loc[averages["site"] == site,"windspeed"].values[0]
        if temp == 0:
            merged_2.iloc[i,temp_col_index] = averages.loc[averages["site"] == site,"temp"].values[0]

    return merged_2

In [27]:
# impute
x_train = impute_na_data(x_train)
x_test = impute_na_data(x_test)

KeyError: 'site'

In [18]:
# concat train and test variables with response variable
train_data = pd.concat([x_train,y_train],axis=1)
test_data = pd.concat([x_test,y_test],axis=1)

In [19]:
train_data.to_csv("../CleanData/full_train_05_09.csv", index = False)
test_data.to_csv("../CleanData/full_test_05_09.csv", index = False)
print("done")

done
